# Dataset mirror (Kaggle)

Re-encodes ASL Citizen at short side 256, so that video decoding stops limiting training.

Preflight found both architectures spending ~52% of every optimizer step waiting on CPU decode. Calibration on 2026-08-11 measured a re-encoded copy **2.61x faster to decode** at 15% of the size, with no frame drift.

## Before running

1. **Add Data** → attach the ASL Citizen mirror.
2. **Accelerator** → **None**. This is CPU-only work and must not spend GPU quota.
3. **Internet** → On, so the repository can be cloned.
4. Resuming a partial build? Also attach this notebook's previous output under **Add Data → Your Work**.

## What to run

| Section | Time | When |
|---|---|---|
| 1-2 setup and manifests | ~2 min | always |
| 3 calibrate | ~15 min | only to re-measure or try another CRF |
| 4 build | ~5.5 h | the actual work |
| 5 verify | ~2 min | always, before trusting the result |

Section 3 is already done. Skip it unless you are changing the encoding settings — otherwise **Run All** spends 15 minutes re-measuring a settled question.

The build is resumable and verifies every clip as it writes it.

## 1. Setup

In [ ]:
import os
import shutil
import subprocess
import sys

assert os.path.exists("/kaggle/input"), (
    "This is the KAGGLE notebook, but this runtime is not Kaggle."
)

REPO_URL = "https://github.com/Adgonzalez2018/ASL-Recognition-Model.git"

# Cloned to /tmp: writable, not part of the saved output, discarded with the
# session. Nothing later has to clean it up.
subprocess.run(["rm", "-rf", "/tmp/asl"], check=True)
subprocess.run(["git", "clone", "-q", REPO_URL, "/tmp/asl"], check=True)
PROJECT = "/tmp/asl/ASL_training"

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", PROJECT, "--no-deps"], check=True
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "av"], check=True)

ARTIFACTS = "/kaggle/working/artifacts"
OUTPUTS = "/kaggle/working/outputs"
WORK = "/tmp/mirror-calibration"

# x264 quality, lower is better. Calibrated at 20; the build must use the
# value that was measured. Defined here rather than in the calibration cell
# so section 4 works when section 3 is skipped.
CRF = 20
os.makedirs(OUTPUTS, exist_ok=True)

DATASET_ROOT = None
for attachment in sorted(os.listdir("/kaggle/input")):
    for root, dirs, _files in os.walk(f"/kaggle/input/{attachment}"):
        if "splits" in dirs and "videos" in dirs:
            DATASET_ROOT = root
            break
    if DATASET_ROOT:
        break
assert DATASET_ROOT, "ASL Citizen not attached. Use Add Data in the sidebar."

assert shutil.which("ffmpeg"), "ffmpeg not found on PATH."


def run(script, **options):
    """Run a project script, streaming its output into this cell.

    Built as an argument list rather than a shell string. IPython's ! escape
    expands $VAR but not {VAR}, and reads $NAME.ext as attribute access, both
    of which have already cost this project a debugging session. subprocess
    has no such rules.
    """
    command = [sys.executable, "-u", f"{PROJECT}/scripts/{script}"]
    for key, value in options.items():
        flag = "--" + key.replace("_", "-")
        # Identity, not equality: 0 == False in Python, so a membership
        # test silently drops --probe-limit 0 and --num-workers 0.
        if value is True:
            command.append(flag)
        elif value is not None and value is not False:
            command += [flag, str(value)]

    print(" ".join(command) + "\n")
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end="")
    code = process.wait()
    if code:
        print(f"\n[exit {code}]")
    return code


print(f"cores      {os.cpu_count()}")
print(f"project    {PROJECT}")
print(f"dataset    {DATASET_ROOT}")

## 2. Manifests

Calibration reads the train manifest to sample clips and to check that frame counts survive re-encoding. `probe_limit=0` skips video probing, so this takes about a minute.

In [ ]:
run(
    "audit_dataset.py",
    dataset_root=DATASET_ROOT,
    output_dir=ARTIFACTS,
    write_manifests=True,
    probe_limit=0,
    expected_classes=2731,
)

## 3. Calibrate

Encodes 300 clips at short side 256, then times decoding through the real loader path — source against mirror, same frame indices, single-threaded so the comparison is fair.

Read three numbers:

1. **DECODE SPEEDUP** — the gate. Under ~1.6x, stop.
2. **projected size** — must fit Kaggle's ~20 GB working directory.
3. **projected encode** — must fit one 12 h CPU session.

`frame mismatches` must be 0. The manifests index against frame counts, so any drift disqualifies the encoding settings.

In [ ]:
run(
    "calibrate_video_mirror.py",
    dataset_root=DATASET_ROOT,
    artifacts_dir=ARTIFACTS,
    work_dir=WORK,
    samples=300,
    decode_samples=120,
    crf=CRF,
    jobs=os.cpu_count(),
    output=f"{OUTPUTS}/mirror_calibration_crf{CRF}.json",
)

## 4. Build the mirror

**Calibration passed on 2026-08-11: 2.61x faster decoding, 7.2 GB projected, 0 frame mismatches.** The cells below carry it out for all 83,399 clips.

This takes about 5.5 hours at 4 cores, so it needs most of a CPU session. It is resumable: a clip already present with the right frame count and geometry is skipped, so re-running after a disconnect continues rather than restarting.

Each clip is verified as it is written — frame count against the source, and short side 256. A failure deletes the clip rather than leaving a partial file that a later run would skip as done.

Leave `CRF` at the calibrated value. Changing it produces a mirror that is not what was measured.

In [ ]:
MIRROR_ROOT = "/kaggle/working/asl_citizen_256"

# Restore a partially built mirror from a previous session's output, so the
# build resumes instead of starting over. Attach it under Add Data -> Your Work.
for attachment in sorted(os.listdir("/kaggle/input")):
    previous = f"/kaggle/input/{attachment}/asl_citizen_256"
    if os.path.isdir(previous) and previous != MIRROR_ROOT:
        print(f"restoring from {previous}")
        subprocess.run(["cp", "-rn", previous, "/kaggle/working/"], check=True)
        break

run(
    "build_video_mirror.py",
    dataset_root=DATASET_ROOT,
    mirror_root=MIRROR_ROOT,
    artifacts_dir=ARTIFACTS,
    crf=CRF,
    jobs=os.cpu_count(),
    output=f"{OUTPUTS}/mirror_build.json",
)

## 5. Verify the mirror

Audits the mirror as if it were the dataset, then compares identities against the source.

The manifest identity hashes sample IDs, paths, labels, signers, and splits — deliberately not resolution or codec. So a correct mirror reproduces the source's hash exactly. That is the proof that the experiment's structure is untouched and only pixels changed.

Both assertions must pass before anything trains against this.

In [ ]:
import json

MIRROR_ARTIFACTS = "/kaggle/working/artifacts-mirror"

# probe_limit=0 here too: the build already verified every clip's frame count
# and geometry, which is a stronger check than the audit's probe.
run(
    "audit_dataset.py",
    dataset_root=MIRROR_ROOT,
    output_dir=MIRROR_ARTIFACTS,
    write_manifests=True,
    probe_limit=0,
    expected_classes=2731,
)

with open(f"{ARTIFACTS}/audits/asl_citizen_audit.json") as handle:
    source = json.load(handle)
with open(f"{MIRROR_ARTIFACTS}/audits/asl_citizen_audit.json") as handle:
    mirrored = json.load(handle)

for name in ("label_map_identity", "manifest_identity"):
    ok = source[name] == mirrored[name]
    print(f"{'OK  ' if ok else 'DIFF'} {name}  {mirrored[name]}")

assert source["manifest_identity"] == mirrored["manifest_identity"], (
    "The mirror's manifest identity differs from the source. Paths or records "
    "changed; it is not a drop-in substitute."
)
assert source["label_map_identity"] == mirrored["label_map_identity"]
print(f"\n{mirrored['counts']['manifest_records']:,} records, identities match")

## 6. Publish

**Save Version → Save & Run All (Commit).** The mirror is ~7 GB in `/kaggle/working` and is lost otherwise.

If the build did not finish inside the session, save anyway, attach this notebook's output under **Add Data → Your Work**, and re-run. Section 4 restores what exists and continues from there.

Once complete, publish `asl_citizen_256` as a Kaggle Dataset and attach it to the training notebook. Training then needs one change: point `DATASET_ROOT` at the mirror. Nothing in `src/` changes, because paths and frame counts are identical.

**All splits must use the same substrate.** The re-encode is lossy, so a baseline trained on the mirror is not comparable to one trained on the source. Mixing them invalidates the comparison.